# EDA dữ liệu đã làm sạch

**Mục tiêu:** mô tả bộ dữ liệu sau làm sạch `data/processed/` và so sánh với dữ liệu nền để kiểm tra việc làm sạch (`scripts/03_clean.py`) chỉ sửa đúng các lỗi đã xác định, không làm thay đổi bức tranh chung.

**Đồng bộ với notebook 01:** mục 1–8 có cùng code với `01_eda_base.ipynb` (dùng chung `src/eda.py`), chỉ khác dữ liệu đầu vào. Mục 9 so sánh trước và sau làm sạch.

**Đầu ra:** thống kê mô tả của dữ liệu dùng cho phân tích (mục 3) và kết quả so sánh trước/sau (mục 9).

## 1. Thiết lập và đọc dữ liệu

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Tìm thư mục gốc dự án (thư mục chứa src/), dù notebook được mở từ đâu
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src import eda

d = eda.load("clean")   # dữ liệu sạch: data/processed/

## 2. Cấu trúc panel

In [ ]:
eda.structure(d)

In [ ]:
# Độ phủ của toàn bộ chỉ tiêu trong indicators.csv
eda.indicator_coverage(d)

**Nhận xét**

- Panel **cân bằng**: 63 tỉnh × 10 năm = 630 dòng, không trùng lặp.
- Chỉ có **63 đơn vị**, là cỡ mẫu nhỏ đối với causal forest. Cần lưu ý khi diễn giải và nên có mô hình cơ sở (hồi quy hiệu ứng cố định) để so sánh.
- Phần lớn trong 1.458 chỉ tiêu chỉ có ở một vài năm hoặc một vài tỉnh, nên panel chỉ chọn 75 chỉ tiêu có độ phủ tốt. Các chỉ tiêu khác vẫn lấy được từ bảng observations khi cần.

## 3. Thống kê mô tả

Thống kê trên toàn bộ các quan sát tỉnh × năm của 16 biến chính, chia theo 5 nhóm: giá trị (3.1–3.3), tương quan giữa các biến (3.4) và biến thiên giữa các tỉnh và trong tỉnh (3.5). Danh sách biến nằm ở `eda.KEY_VARS` trong `src/eda.py`; khi nhóm chốt Y, W, X thì bổ sung các biến đó vào danh sách.

In [ ]:
# 3.1 Toàn bộ giai đoạn 2015–2024
eda.describe(d)

In [ ]:
# 3.2 Trung bình theo giai đoạn, ứng với hai ấn phẩm nguồn
eda.describe_by_period(d)

In [ ]:
# 3.3 Trung bình theo vùng (mã vùng ở mục 2)
eda.describe_by_region(d)

**Nhận xét**

- Bảng thống kê mô tả của 15/16 biến chính **giống hệt dữ liệu nền**, vì làm sạch không chạm vào các biến này.
- Khác biệt duy nhất là **học sinh trên một giáo viên**: sau khi sửa số học sinh năm 2018, giá trị lớn nhất giảm từ 70,1 xuống 29,8 và skew giảm từ 3,9 xuống 0,2. Phân phối của biến này giờ gần đối xứng.
- Các nhận xét khác giữ nguyên như notebook 01: phân phối lệch phải mạnh ở các biến quy mô (nên lấy log), chênh lệch lớn giữa giai đoạn và giữa các vùng.
- Bảng 3.1 là **bảng thống kê mô tả dùng cho bài viết**.

### 3.4 Ma trận tương quan

Hệ số tương quan r giữa từng cặp biến chính (từ −1 đến +1; |r| ≥ 0,7 coi là mạnh). Tính theo hai cách:

- **Pearson:** đo quan hệ tuyến tính, dễ bị vài giá trị cực đoan (Hà Nội, TP. Hồ Chí Minh) chi phối.
- **Spearman:** tính trên thứ hạng của các quan sát nên ít bị ngoại lai chi phối, phù hợp với các biến lệch phải mạnh như ở mục 3.1.

Tương quan chỉ cho biết hai biến có đi cùng nhau hay không, **không phải quan hệ nhân quả**.

In [ ]:
# Pearson: bản đồ nhiệt và các cặp |r| ≥ 0,7
eda.correlation(d)

In [ ]:
# Spearman: chỉ liệt kê các cặp |r| ≥ 0,7
eda.correlation(d, method="spearman", plot=False)

**Nhận xét**

- **GRDP bình quân đầu người và năng suất lao động** có r = 0,96 theo cả hai cách, gần như đo cùng một thứ. Khi chọn biến kiểm soát (X) chỉ nên đưa một trong hai vào mô hình.
- Theo Pearson chỉ có 4 cặp |r| ≥ 0,7; theo Spearman có 8 cặp. Spearman cho thấy một **cụm biến "mức độ phát triển"** liên quan chặt với nhau: GRDP bình quân đầu người, năng suất lao động, thu nhập bình quân, tỷ lệ hộ nghèo (r = −0,92 với thu nhập), mật độ dân số. Pearson đánh giá thấp các quan hệ này vì phân phối lệch. Khi chọn X nên xem cả hai cách và tránh đưa nhiều biến cùng cụm vào mô hình.
- **Học sinh trên một giáo viên và thu nhập bình quân** (Pearson r = 0,71): các tỉnh thu nhập cao, đô thị hóa cao có lớp học đông hơn. Đây là tương quan, không có nghĩa thu nhập làm tăng sĩ số lớp.
- Các cặp biến còn lại có |r| < 0,7, chưa có dấu hiệu trùng lặp nghiêm trọng giữa các nhóm biến.

### 3.5 Biến thiên giữa các tỉnh và trong từng tỉnh

Với dữ liệu panel, sự khác nhau của một biến đến từ hai nguồn:

- **Giữa các tỉnh:** tỉnh này luôn cao hơn tỉnh kia (ví dụ Hà Nội luôn đông dân hơn Bắc Kạn).
- **Trong từng tỉnh:** cùng một tỉnh nhưng thay đổi qua các năm.

Mô hình có **hiệu ứng cố định tỉnh và năm** chỉ học được từ phần biến thiên trong tỉnh sau khi bỏ xu hướng chung của cả nước theo năm (cột cuối của bảng). Biến nào có phần này quá nhỏ thì mô hình gần như không có thông tin để ước lượng tác động của biến đó. Cách tính tương tự lệnh `xtsum` của Stata.

In [ ]:
eda.variation(d)

**Nhận xét**

- Bảng giống notebook 01, trừ **học sinh trên một giáo viên**: phần phương sai trong tỉnh giảm từ 47% xuống 33%, và phần còn lại sau khi bỏ hiệu ứng cố định tỉnh và năm giảm từ 31,7% xuống 9,3%.
- Nghĩa là trước khi làm sạch, **lỗi số liệu năm 2018 tạo ra biến thiên giả trong tỉnh**. Mô hình có hiệu ứng cố định sẽ coi nhầm đó là thay đổi thật qua thời gian. Đây là ví dụ cụ thể cho thấy vì sao phải làm sạch trước khi chạy mô hình.
- Các nhận xét khác giữ nguyên như notebook 01: dân số, mật độ, đô thị hóa, hộ nghèo gần như không đổi trong tỉnh (phù hợp làm X hoặc biến phân nhóm); chỉ số phát triển GRDP, vốn FDI/GRDP, thất nghiệp, vốn đầu tư/GRDP biến thiên nhiều trong tỉnh (ứng viên cho Y hoặc W).

## 4. Thiếu dữ liệu

In [ ]:
# Tỷ lệ ô trống theo biến
missing = eda.missing_by_var(d)

In [ ]:
# Số tỉnh thiếu theo biến × năm
eda.missing_heatmap(d, missing.index)

**Nhận xét.** Giống notebook 01, chỉ thêm các ô được **bỏ trống có chủ đích** khi làm sạch: thuế sản phẩm (5 ô), số trường (2 ô), số học sinh và học sinh trên một giáo viên (mỗi biến 2 ô). Các ô này là số liệu sai trong ấn phẩm và không suy ra được giá trị đúng; để trống tốt hơn giữ số sai. Danh sách đầy đủ ở mục 9.

## 5. Cờ chất lượng

In [ ]:
# 5.1 Số quan sát là số sơ bộ theo năm
eda.preliminary_by_year(d)

In [ ]:
# 5.2 Cờ series_break (chuỗi gián đoạn so với năm trước)
break_count = eda.breaks(d)

**Nhận xét**

- **Số sơ bộ** chủ yếu rơi vào năm 2024 (năm cuối của ấn phẩm mới nhất). Đề xuất kiểm định độ vững bằng cách **bỏ năm 2024**.
- **Năm 2020 cả 63 tỉnh đều có cờ gián đoạn**, vì là năm chuyển từ ấn phẩm 2015–2021 sang ấn phẩm 2020–2024. Mô hình cần có hiệu ứng cố định năm hoặc biến `post2020`.
- Cột `has_break` tính trên cả 75 biến nên rất rộng; khi lọc chỉ nên xét đúng các biến của mô hình (cột `break_vars`).
- Nhóm FDI bị gắn cờ nhiều nhất, phù hợp với đặc điểm biến động mạnh của dòng vốn FDI.
- Cờ `series_break` được giữ nguyên như dữ liệu nền (cờ mô tả chuỗi gốc, không tính lại sau làm sạch).

## 6. Kiểm tra tính hợp lý

In [ ]:
# 6.1 Giá trị âm
eda.negative_values(d)

In [ ]:
# 6.2 Biến tỷ lệ % nằm ngoài khoảng 0–100
eda.pct_out_of_range(d)

In [ ]:
# 6.3 Ràng buộc logic giữa các chỉ tiêu (số dòng vi phạm, kèm danh sách)
eda.logic_summary(d)

In [ ]:
# 6.4 Xem kỹ các tỉnh có thành phần GRDP lệch tổng ở dữ liệu nền
cols = ["province_name", "year", "grdp_current"] + eda.GRDP_PARTS
d["panel"].query("province_name in ['Vinh Long', 'Ha Nam', 'Bac Giang'] and (year <= 2018 or year >= 2023)")[cols]

**Nhận xét**

- Ràng buộc **thành phần GRDP = tổng GRDP** giờ thỏa mãn ở mọi dòng (dữ liệu nền: 6 dòng vi phạm). Bảng 6.4 cho thấy Vĩnh Long 2024 đã có GRDP dịch vụ 33.206,1; thuế sản phẩm của Hà Nam 2015–2018 và Bắc Giang 2016 để trống.
- **Lao động 3 khu vực lệch tổng** vẫn còn ở Tây Ninh 2019 và Đồng Tháp 2022. Ấn phẩm in đúng như vậy và không xác định được số nào sai, nên giữ nguyên và ghi nhận trong nhật ký.
- **Tăng trưởng GRDP tự tính lệch chỉ số GSO** vẫn còn 13 dòng. Đây là mâu thuẫn trong chính ấn phẩm, không sửa; khi cần tốc độ tăng trưởng thì dùng `grdp_growth_index`.
- Giá trị âm và tỷ lệ tiêm chủng > 100% giữ nguyên như notebook 01.

## 7. Bước nhảy và đỉnh bất thường giữa các năm

Hai cách phát hiện bổ sung cho nhau:

- **Bước nhảy:** giá trị năm t gấp ≥ 5 lần hoặc chỉ còn ≤ 1/5 năm t−1. Bắt được các thay đổi rất lớn.
- **Đỉnh một năm:** với các biến quy mô (số người, số cơ sở, số thuê bao…), giá trị năm t lệch hơn 25% so với trung bình năm t−1 và t+1, trong khi hai năm đó gần nhau. Bắt được các lỗi nhỏ hơn mà cách thứ nhất bỏ sót.

In [ ]:
# 7.1 Bước nhảy ≥ 5 lần
eda.jumps(d)

In [ ]:
# 7.2 Đỉnh một năm ở các biến quy mô
eda.spikes(d)

In [ ]:
# 7.3 Ví dụ: học sinh trên một giáo viên (đường đỏ: TP. Hồ Chí Minh)
eda.lines(d, "students_per_teacher", "Học sinh trên một giáo viên", highlight="Ho Chi Minh City")

In [ ]:
# 7.4 Ví dụ: thuê bao internet trên 100 dân (đường đỏ: Yên Bái)
eda.lines(d, "internet_per_100", "Thuê bao internet trên 100 dân", highlight="Yen Bai")

**Nhận xét**

- Bước nhảy của GRDP dịch vụ và số trường đã hết; thuê bao cố định và di động giảm từ 11 xuống 9 lần (Gia Lai 2019 đã đổi lại chỗ).
- Đỉnh một năm giảm **từ 38 xuống 25**. Các đỉnh còn lại là những trường hợp đã xem xét ở notebook 01 và có thể là biến động thật.
- Biểu đồ 7.3: chuỗi học sinh trên một giáo viên của TP. Hồ Chí Minh không còn đỉnh 70 năm 2018 (năm này để trống).
- Thuê bao internet (biểu đồ 7.4) **chưa xử lý**, vẫn như dữ liệu nền; chờ nhóm quyết định có dùng biến này không.

## 8. Diễn biến theo thời gian và ngoại lai

In [ ]:
# Mỗi đường mảnh là một tỉnh, đường đen là trung vị 63 tỉnh
eda.trends(d, ["grdp_pc", "invest_to_grdp", "invest_fdi_to_grdp",
               "budget_rev_domestic_to_grdp", "trained_labor_rate", "urbanization_rate"])

In [ ]:
# Các giá trị cao nhất: lỗi hay thực tế?
eda.top_values(d, ["invest_to_grdp", "invest_fdi_to_grdp", "budget_rev_domestic_to_grdp"])

**Nhận xét**

- **Ranh giới năm 2020:** tỷ lệ đô thị hóa có bước tăng bậc ngay năm 2020, đúng năm chuyển ấn phẩm (đồng thời là năm áp dụng kết quả Tổng điều tra dân số 2019); tỷ lệ lao động qua đào tạo tăng nhanh bất thường giai đoạn 2019–2021. Đây là gián đoạn do nguồn số liệu, hiệu ứng cố định năm sẽ hấp thụ phần lớn.
- **Vốn đầu tư/GRDP > 100%** ở Hà Tĩnh, Thái Nguyên, Thanh Hóa giai đoạn 2015–2016 trùng với thời kỳ triển khai các dự án FDI rất lớn tại các tỉnh này (khu liên hợp gang thép, tổ hợp điện tử, lọc hóa dầu). Đây là **số liệu thật**, giữ nguyên, nhưng cần kiểm định độ vững khi bỏ các quan sát này.
- **Hà Nội, TP. Hồ Chí Minh** vượt trội ở hầu hết các biến quy mô. Đây là ngoại lai thật, không xóa.

## 9. So sánh trước và sau làm sạch

In [ ]:
before = eda.load("base")   # dữ liệu nền để so sánh

# 9.1 Nhật ký làm sạch: mọi ô đã sửa, bỏ trống, tính lại hoặc đã xem xét và giữ nguyên
corrections = pd.read_csv(eda.PROCESSED / "corrections.csv")
corrections.action.value_counts()

In [ ]:
corrections[["province_name", "year", "variable", "value_before", "value_after", "action", "evidence"]]

In [ ]:
# 9.2 Số ô thay đổi theo biến
eda.changed_vars(before, d)

In [ ]:
# 9.3 Số dòng vi phạm ràng buộc logic, trước và sau
eda.compare_logic(before, d)

In [ ]:
# 9.4 Số bước nhảy và đỉnh một năm, trước và sau
pd.DataFrame({"Trước": [int(eda.jumps(before).sum().iloc[0]), len(eda.spikes(before))],
              "Sau": [int(eda.jumps(d).sum().iloc[0]), len(eda.spikes(d))]},
             index=["Bước nhảy ≥ 5 lần", "Đỉnh một năm (biến quy mô)"])

In [ ]:
# 9.5 Thống kê mô tả của các biến bị thay đổi, trước và sau
eda.compare_describe(before, d, ["students", "students_per_teacher", "schools", "grdp_current_serv",
                                 "grdp_current_tax", "fixed_phone_subs", "mobile_subs"])

**Nhận xét**

- Làm sạch chỉ thay đổi **28 ô** trên tổng số 54.810 ô số liệu của panel (630 dòng × 87 biến), gồm 7 biến, tất cả đều có trong nhật ký kèm lý do và trang PDF. Ngoài ra có 2 ô được xem xét và giữ nguyên (lao động Tây Ninh 2019, Đồng Tháp 2022).
- Các kiểm tra đều tốt lên: thành phần GRDP lệch tổng từ 6 dòng còn 0; đỉnh một năm từ 38 còn 25; các bước nhảy do lỗi đã hết.
- Thống kê mô tả của các biến chính (mục 3) **không thay đổi**, trừ học sinh trên một giáo viên. Nghĩa là làm sạch sửa lỗi cục bộ, **không làm thay đổi bức tranh chung** của dữ liệu.
- Với các biến bị sửa (bảng 9.5), trung bình thay đổi ít nhưng giá trị lớn nhất và độ lệch (skew) giảm rõ ở các biến có lỗi cực đoan: học sinh trên một giáo viên (skew 3,9 → 0,2), số trường (9,1 → 2,9), số học sinh (4,8 → 3,9). Các biến GRDP và thuê bao gần như không đổi vì lỗi chỉ ở một vài ô so với độ lớn của cả chuỗi.

## 10. Kết luận

- Dữ liệu sạch (`data/processed/panel_clean.csv`) **sẵn sàng cho bước chuẩn bị dữ liệu mô hình**.
- Những điểm còn lại **không xử lý bằng sửa số** mà bằng thiết kế mô hình: gián đoạn năm 2020 (hiệu ứng cố định năm, `post2020`), phân phối lệch (lấy log), ngoại lai thật và số sơ bộ (kiểm định độ vững).
- Cần nhóm quyết định: có dùng thuê bao internet không (phạm vi thống kê khác nhau trước 2020).
- Khi chạy mô hình, chạy trên **cả dữ liệu nền và dữ liệu sạch** để kiểm tra kết quả không phụ thuộc vào bước làm sạch.